In [ ]:
import pandas as pd
import numpy as np

excel_file = '225 g of bacon on gas burner, fan off.xls'

print("1. Loading tabs from Excel...")
df_vel = pd.read_excel(excel_file, sheet_name='Velocity')
df_sens = pd.read_excel(excel_file, sheet_name='Sensors')
df_inst = pd.read_excel(excel_file, sheet_name='Instruments')

time_col_vel = next(col for col in df_vel.columns if 'time' in col.lower())
time_col_sens = next(col for col in df_sens.columns if 'time' in col.lower())
time_col_inst = next(col for col in df_inst.columns if 'time' in col.lower())

df_vel.rename(columns={time_col_vel: 'Time'}, inplace=True)
df_sens.rename(columns={time_col_sens: 'Time'}, inplace=True)
df_inst.rename(columns={time_col_inst: 'Time'}, inplace=True)

# Force them all to be floats so the int/float warning disappears
df_vel['Time'] = df_vel['Time'].astype(float).round(1)
df_sens['Time'] = df_sens['Time'].astype(float).round(1)
df_inst['Time'] = df_inst['Time'].astype(float).round(1)

print("2. Merging data...")
# Merge them together
merged_df = df_vel.merge(df_sens, on='Time', how='outer').merge(df_inst, on='Time', how='outer')

# Sort by time, Forward Fill, AND Backward Fill
merged_df = merged_df.sort_values('Time').ffill().bfill()
merged_df['class_id'] = np.where(merged_df['Time'] < 0, 0, 1)
merged_df['label'] = merged_df['class_id'].map({0: 'Normal', 1: 'Cooking', 2: 'Other'})

actual_temp = next((col for col in merged_df.columns if 't1' in col.lower() or 'temp' in col.lower()), None)
actual_hum = next((col for col in merged_df.columns if 'humid' in col.lower() or 'rh' in col.lower()), None)
actual_co = next((col for col in merged_df.columns if 'co ' in col.lower() or 'co(' in col.lower() or 'co_' in col.lower() or '- co' in col.lower()), None)
actual_co2 = next((col for col in merged_df.columns if 'co2' in col.lower()), None)

# --- DEBUGGING PRINTS ---
print("\n--- COLUMN SEARCH RESULTS ---")
print(f"Temperature column found: {actual_temp}")
print(f"Humidity column found: {actual_hum}")
print(f"CO (TVOC) column found: {actual_co}")
print(f"CO2 (eCO2) column found: {actual_co2}")
print("-----------------------------\n")

final_df = pd.DataFrame({
    'label': merged_df['label'],
    'class_id': merged_df['class_id'],
    'temperature': merged_df[actual_temp] if actual_temp else np.nan,
    'humidity': merged_df[actual_hum] if actual_hum else np.nan,
    'tvoc_ppb': merged_df[actual_co] if actual_co else np.nan, 
    'eco2_ppm': merged_df[actual_co2] if actual_co2 else np.nan
})

cols_to_fix = ['temperature', 'humidity', 'tvoc_ppb', 'eco2_ppm']
final_df[cols_to_fix] = final_df[cols_to_fix].apply(pd.to_numeric, errors='coerce')

if final_df['eco2_ppm'].max() < 10:
    final_df['eco2_ppm'] = final_df['eco2_ppm'] * 10000

# TEMPORARILY DISABLED DROPNA TO SEE THE PROBLEM
# final_df = final_df.dropna(subset=['tvoc_ppb', 'eco2_ppm'])

output_file = 'NIST_cleaned_bacon.csv'
final_df.to_csv(output_file, index=False)

print(f"File saved to {output_file} WITHOUT dropping empty rows.")
print("\nHere are the first 5 rows:")
print(final_df.head())

1. Loading tabs from Excel...
2. Merging data...

--- COLUMN SEARCH RESULTS ---
Temperature column found: T1 (deg C)
Humidity column found: Rel Humidity (%)
CO (TVOC) column found: A - CO (ppm)
CO2 (eCO2) column found: None
-----------------------------

File saved to NIST_cleaned_bacon.csv WITHOUT dropping empty rows.

Here are the first 5 rows:
    label  class_id  temperature  humidity  tvoc_ppb  eco2_ppm
0  Normal         0        23.28       NaN       NaN       NaN
1  Normal         0        23.24       NaN       NaN       NaN
2  Normal         0        23.17       NaN       NaN       NaN
3  Normal         0        23.11       NaN       NaN       NaN
4  Normal         0        23.05       NaN       NaN       NaN


C:\Users\guiga\AppData\Local\Temp\ipykernel_32304\3741679281.py:24: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df = df_vel.merge(df_sens, on='Time', how='outer').merge(df_inst, on='Time', how='outer')
